# CS5228 Project — HDB Rental Rate Prediction

Team 9. Angeles Mathena (A0351915A), Alzamil Nojood Ahmad (A0350755B), Su Yubo (A0356908N), Szentpeteri David (A0357858E), Dai Peisong (A0351174J).

In [19]:
#IMPORTS + GLOBAL CONST
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import BallTree

pd.set_option('display.max_columns', None)
EARTH_RADIUS_KM = 6371

In [20]:
#LOAD CORE TRAIN/TEST (READ_CSV) - RAW COMPETITION DATA, TARGET=MONTHLY_RENT
df_train = pd.read_csv('data/train.csv')
df_test = pd.read_csv('data/test.csv')

print('train:', df_train.shape, '| test:', df_test.shape)

train: (150000, 11) | test: (50000, 10)


In [21]:
#LOAD AUX TABLES (READ_CSV) - GEO/LANDMARK/MACRO SRC FOR FEAT ENG
df_hdb_block = pd.read_csv('data/auxiliary/sg-hdb-block.csv')
df_mrt = pd.read_csv('data/auxiliary/sg-mrt-stations.csv')
df_mall = pd.read_csv('data/auxiliary/sg-shopping-malls.csv')
df_school = pd.read_csv('data/auxiliary/sg-schools.csv')
df_coe = pd.read_csv('data/auxiliary/sg-coe-prices.csv')
df_stock = pd.read_csv('data/auxiliary/sg-stock-prices.csv')

In [22]:
#CLEAN (NORMALIZE CATS, DROP ZERO-VARIANCE COLS)
#WHY: "4 ROOM"/"4-ROOM" ARE 1 CAT SPLIT IN 2 BY FORMATTING - FEE/FURNISHED ARE CONST ACROSS ALL ROWS (KNOWN FROM PRIOR EDA), 0 SIGNAL
#FLAT_TYPE CASING NORMALIZE + DROP CONST COLS (FEE, FURNISHED)
def clean(df):
    df = df.copy()
    df['FLAT_TYPE'] = df['FLAT_TYPE'].str.replace(' ', '-', regex=False)
    df['STREET'] = df['STREET'].str.lower().str.strip()

    return df.drop(columns=['FEE', 'FURNISHED'])

df_train = clean(df_train)
df_test = clean(df_test)

In [23]:
#TEMPORAL FEATS (MTH SINCE START, MTH-OF-YR, LEASE AGE)
#WHY: 2021-2026 RENT SHOWS STRONG UPWARD TREND & TEST SET IS FUTURE-ONLY --> MODEL MUST EXTRAPOLATE ON AN ORDINAL TIME TERM, NOT JUST INTERPOLATE
for df in (df_train, df_test):
    df['RENT_APPROVAL_DATE'] = pd.to_datetime(df['RENT_APPROVAL_DATE'], format='%Y-%m')
    df['MTH_SINCE_START'] = (df['RENT_APPROVAL_DATE'].dt.year - 2021) * 12 + df['RENT_APPROVAL_DATE'].dt.month
    df['MTH_OF_YR'] = df['RENT_APPROVAL_DATE'].dt.month
    df['LEASE_AGE_YR'] = df['RENT_APPROVAL_DATE'].dt.year - df['LEASE_COMMENCE_DATE']

In [24]:
#GEO JOIN (MERGE ON TOWN+BLOCK+STREET)
#WHY: LAT/LON REQ FOR MRT/MALL/SCHOOL DIST FEATS BELOW - ONLY AVAIL JOIN KEY. ~80% MATCH (KNOWN FROM PRIOR EDA) - REST STAY NaN, HISTGB HANDLES NATIVELY, NO IMPUTE NEEDED
join_cols = ['TOWN', 'BLOCK', 'STREET']
df_train = df_train.merge(df_hdb_block[join_cols + ['LATITUDE', 'LONGITUDE']], on=join_cols, how='left')
df_test = df_test.merge(df_hdb_block[join_cols + ['LATITUDE', 'LONGITUDE']], on=join_cols, how='left')

print(f'geo match train: {df_train["LATITUDE"].notna().mean():.1%} | test: {df_test["LATITUDE"].notna().mean():.1%}')

geo match train: 100.0% | test: 100.0%


In [25]:
#DIST HELPER (1 FN, REUSED FOR MRT/MALL/SCHOOL BELOW)
#NEAREST LANDMARK DIST (BALLTREE, HAVERSINE) - KM
def nearest_dist_km(src_lat, src_lon, ref_lat, ref_lon):
    valid = src_lat.notna() & src_lon.notna()
    dist_km = pd.Series(np.nan, index=src_lat.index)

    if valid.sum() == 0:
        return dist_km

    ref_rad = np.radians(np.column_stack([ref_lat, ref_lon]))
    tree = BallTree(ref_rad, metric='haversine')
    src_rad = np.radians(np.column_stack([src_lat[valid], src_lon[valid]]))
    dist_rad, _ = tree.query(src_rad, k=1)
    dist_km.loc[valid] = dist_rad[:, 0] * EARTH_RADIUS_KM

    return dist_km

In [26]:
#DIST TO NEAREST MRT/MALL/PRIMARY SCHOOL
#WHY: PROXIMITY TO AMENITIES = CLASSIC SG HOUSING PRICE DRIVER (PER PROJECT SPEC) - SCHOOLS FILTERED TO PRIMARY ONLY, SPEC ASKS FOR PRIMARY NOT SECONDARY
df_school_primary = df_school[df_school['MAINLEVEL_CODE'] == 'primary']

for df in (df_train, df_test):
    df['DIST_MRT_KM'] = nearest_dist_km(df['LATITUDE'], df['LONGITUDE'], df_mrt['LATITUDE'], df_mrt['LONGITUDE'])
    df['DIST_MALL_KM'] = nearest_dist_km(df['LATITUDE'], df['LONGITUDE'], df_mall['LATITUDE'], df_mall['LONGITUDE'])
    df['DIST_SCH_KM'] = nearest_dist_km(df['LATITUDE'], df['LONGITUDE'], df_school_primary['LATITUDE'], df_school_primary['LONGITUDE'])

In [27]:
#COE MONTHLY MACRO FEAT (MEAN PREMIUM ACROSS CAT+ROUND)
#WHY: SPEC FLAGS COE AS ECON INDICATOR OF RENT "CLIMATE" - COLLAPSE TO 1 SIGNAL/MTH, JOIN ON MTH_SINCE_START
mth_num = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
df_coe['QUOTA_PREMIUM'] = df_coe['QUOTA_PREMIUM'].str.replace(r'[\$,]', '', regex=True).astype(float)
df_coe['MTH_SINCE_START'] = (df_coe['YEAR'] - 2021) * 12 + df_coe['MONTH'].map(mth_num)
coe_monthly = df_coe.groupby('MTH_SINCE_START')['QUOTA_PREMIUM'].mean().reset_index(name='COE_PREMIUM_AVG')

df_train = df_train.merge(coe_monthly, on='MTH_SINCE_START', how='left')
df_test = df_test.merge(coe_monthly, on='MTH_SINCE_START', how='left')

print(f'COE match train: {df_train["COE_PREMIUM_AVG"].notna().mean():.1%} | test: {df_test["COE_PREMIUM_AVG"].notna().mean():.1%}')

COE match train: 100.0% | test: 100.0%


In [28]:
#STOCK MONTHLY MACRO FEAT (MEAN CLOSE ACROSS ALL CO.)
#WHY: SPEC FLAGS STOCK PRICES AS GEN ECON INDICATOR - SAME COLLAPSE-TO-1-SIGNAL APPROACH AS COE
df_stock['DATE'] = pd.to_datetime(df_stock['DATE'])
df_stock['MTH_SINCE_START'] = (df_stock['DATE'].dt.year - 2021) * 12 + df_stock['DATE'].dt.month
stock_monthly = df_stock.dropna(subset=['CLOSE']).groupby('MTH_SINCE_START')['CLOSE'].mean().reset_index(name='STOCK_CLOSE_AVG')

df_train = df_train.merge(stock_monthly, on='MTH_SINCE_START', how='left')
df_test = df_test.merge(stock_monthly, on='MTH_SINCE_START', how='left')

print(f'stock match train: {df_train["STOCK_CLOSE_AVG"].notna().mean():.1%} | test: {df_test["STOCK_CLOSE_AVG"].notna().mean():.1%}')

stock match train: 100.0% | test: 100.0%


In [29]:
#ASSEMBLE FEAT LIST + CAST CAT DTYPES
#WHY: HISTGB NATIVE CAT SUPPORT NEEDS PANDAS 'CATEGORY' DTYPE - UNION TRAIN+TEST CATS SO NO UNSEEN-CAT MISMATCH AT PREDICT TIME. STREET/BLOCK STAY OUT OF THE FEAT LIST (NEAR-UNIQUE, WOULD OVERFIT) BUT ARE STILL NEEDED AS JOIN KEYS FOR THE BLOCK-HISTORY FEAT BELOW
#X IS BUILT AFTER THE SPLIT NOW - HIST_BLK MUST ONLY EVER SEE PAST ROWS, SO IT CANNOT BE COMPUTED ON THE WHOLE FRAME UP HERE
num_feats = ['FLOOR_AREA_SQM', 'LEASE_AGE_YR', 'MTH_SINCE_START', 'MTH_OF_YR', 'DIST_MRT_KM', 'DIST_MALL_KM', 'DIST_SCH_KM', 'COE_PREMIUM_AVG', 'STOCK_CLOSE_AVG']
cat_feats = ['TOWN', 'FLAT_TYPE', 'FLAT_MODEL']
hist_feats = ['HIST_BLK', 'HIST_BLK_N']

for col in cat_feats:
    all_cats = pd.concat([df_train[col], df_test[col]]).unique()
    df_train[col] = pd.Categorical(df_train[col], categories=all_cats)
    df_test[col] = pd.Categorical(df_test[col], categories=all_cats)

feat_cols = num_feats + cat_feats + hist_feats

In [30]:
#CHRONOLOGICAL HOLDOUT SPLIT (LAST N CALENDAR MTHS)
#WHY: KAGGLE TEST.CSV IS ITSELF A FUTURE-ONLY SPLIT OF TRAIN.CSV - RANDOM SPLIT WOULD LEAK FUTURE ROWS INTO VAL & OVERSTATE SCORE
#WHY MTHS NOT 15%: TEST.CSV RUNS 16 MTHS PAST TRAIN END, THE OLD ROW-% CUT ONLY SPANNED 8 - AND IT SLICED ONE MTH DOWN THE MIDDLE, PUTTING THE SAME BLOCK+MTH ON BOTH SIDES
holdout_months = 12
df_train = df_train.sort_values('RENT_APPROVAL_DATE').reset_index(drop=True)

months = df_train['RENT_APPROVAL_DATE'].drop_duplicates().sort_values()
cut_date = months.iloc[-holdout_months]
is_hold = df_train['RENT_APPROVAL_DATE'] >= cut_date
df_search, df_hold = df_train[~is_hold], df_train[is_hold]

print(f'search : {len(df_search)} rows, up to {df_search["RENT_APPROVAL_DATE"].max():%Y-%m}')
print(f'holdout: {len(df_hold)} rows, {cut_date:%Y-%m} -> {df_train["RENT_APPROVAL_DATE"].max():%Y-%m} (horizon 1-{holdout_months} mths, real test.csv is 1-16)')

search : 114842 rows, up to 2024-03
holdout: 35158 rows, 2024-04 -> 2025-03 (horizon 1-12 mths, real test.csv is 1-16)


In [31]:
#LAGGED BLOCK HISTORY (MEAN RENT OF SAME BLOCK+FLAT_TYPE, PAST ROWS ONLY)
#WHY: "WHAT DID THE IDENTICAL FLAT NEXT DOOR GO FOR" IS THE STRONGEST SIGNAL AVAILABLE - BUT IT IS BUILT FROM THE TARGET, SO IT LEAKS UNLESS EVERY VALUE COMES FROM ROWS THE MODEL WOULD REALLY HAVE HAD
#is_future=True  -> TARGET SITS ENTIRELY AFTER SOURCE (holdout, test.csv): USE ALL OF SOURCE. MIRRORS REALITY, WHERE THE NEWEST LABEL IS ALREADY 1-16 MTHS OLD AT PREDICT TIME
#is_future=False -> TARGET *IS* SOURCE (the train rows): EXPANDING MEAN, shift(1) DROPS THE CURRENT MTH SO A ROW NEVER SEES ITSELF NOR ITS OWN MTH'S SIBLINGS
#GRAIN PICKED BY ABLATION: BLOCK ALONE BEAT BLOCK+STREET+TOWN - COARSER GRAINS ONLY RESTATE THE TOWN/FLAT_TYPE CATS AND DILUTE THE SPLITS
hist_keys = ['STREET', 'BLOCK', 'FLAT_TYPE']

def block_history(source, target, is_future):
    if is_future:
        stats = source.groupby(hist_keys, observed=True)['MONTHLY_RENT'].agg(HIST_BLK='mean', HIST_BLK_N='size').reset_index()
        out = target[hist_keys].merge(stats, on=hist_keys, how='left')
    else:
        by_mth = source.groupby(hist_keys + ['MTH_SINCE_START'], observed=True)['MONTHLY_RENT'].agg(s='sum', c='size').reset_index().sort_values('MTH_SINCE_START')
        by_mth[['cs', 'cc']] = by_mth.groupby(hist_keys, observed=True)[['s', 'c']].cumsum()
        by_mth[['cs', 'cc']] = by_mth.groupby(hist_keys, observed=True)[['cs', 'cc']].shift(1)
        by_mth['HIST_BLK'], by_mth['HIST_BLK_N'] = by_mth['cs'] / by_mth['cc'], by_mth['cc']
        out = target[hist_keys + ['MTH_SINCE_START']].merge(by_mth[hist_keys + ['MTH_SINCE_START'] + hist_feats], on=hist_keys + ['MTH_SINCE_START'], how='left')

    return out['HIST_BLK'].to_numpy(), out['HIST_BLK_N'].to_numpy()

def build_X(target, source, is_future):
    X = target[num_feats + cat_feats].copy()
    X['HIST_BLK'], X['HIST_BLK_N'] = block_history(source, target, is_future)

    return X[feat_cols]

X_search, y_search = build_X(df_search, df_search, False), df_search['MONTHLY_RENT']
X_hold, y_hold = build_X(df_hold, df_search, True), df_hold['MONTHLY_RENT']

print(f'HIST_BLK coverage search: {X_search["HIST_BLK"].notna().mean():.1%} | holdout: {X_hold["HIST_BLK"].notna().mean():.1%}')

HIST_BLK coverage search: 86.1% | holdout: 98.8%


In [32]:
#HYPERPARAM SEARCH (RANDOMIZEDSEARCHCV + TIMESERIESSPLIT)
#WHY: TIMESERIESSPLIT KEEPS EACH CV FOLD CHRONOLOGICAL (NO FUTURE-LEAK) - HOLDOUT SET STAYS FULLY UNTOUCHED FOR THE FINAL UNBIASED SCORE
param_dist = {
    'learning_rate': [0.03, 0.05, 0.1, 0.15],
    'max_leaf_nodes': [15, 31, 63],
    'max_iter': [100, 200, 300],
    'min_samples_leaf': [20, 50, 100],
    'l2_regularization': [0.0, 0.1, 1.0],
}
base_model = HistGradientBoostingRegressor(categorical_features='from_dtype', random_state=42)
search = RandomizedSearchCV(base_model, param_dist, n_iter=8, cv=TimeSeriesSplit(n_splits=4), scoring='neg_root_mean_squared_error', random_state=42, n_jobs=-1)
search.fit(X_search, y_search)

print('best params:', search.best_params_)
print(f'best CV RMSE: {-search.best_score_:.2f}')

best params: {'min_samples_leaf': 20, 'max_leaf_nodes': 15, 'max_iter': 300, 'learning_rate': 0.1, 'l2_regularization': 0.1}
best CV RMSE: 535.37


In [33]:
#HOLDOUT EVAL (UNBIASED - NEVER SEEN DURING SEARCH)
best_model = search.best_estimator_
hold_pred = best_model.predict(X_hold)
rmse_hold = root_mean_squared_error(y_hold, hold_pred)

print(f'holdout RMSE: {rmse_hold:.2f}')
print('public leaderboard benchmark (captured 2026-08-31, "CS5228 Rent Kakis"): 441.63')
print('prior naive linear-regression baseline: 700.33')

# INITIAL RMSE: 464.79
# RMSE after 100%geo: 462.12
# --- HOLDOUT REDEFINED HERE: LAST 12 MTHS, NOT LAST 15% OF ROWS (WAS 8 MTHS) ---
# THE TWO NUMBERS ABOVE ARE ON THE OLD, EASIER SPLIT - NOT COMPARABLE TO THE ONES BELOW
# RMSE on 12-mth holdout, no block history: 481.59
# RMSE on 12-mth holdout, with block history: 467.75
# ACTUAL PUBLIC LB WITH THIS MODEL: 436.05 (HOLDOUT READS ~30 PTS HARDER: LESS TRAIN DATA, LONGER HORIZON)

holdout RMSE: 467.75
public leaderboard benchmark (captured 2026-08-31, "CS5228 Rent Kakis"): 441.63
prior naive linear-regression baseline: 700.33


In [34]:
#PERMUTATION IMPORTANCE (INTERPRETATION)
#WHY: HISTGB HAS NO NATIVE FEATURE_IMPORTANCES_ - PERMUTATION IMPORTANCE ON HOLDOUT GIVES MODEL-AGNOSTIC, LEAK-FREE RANKING
perm = permutation_importance(best_model, X_hold, y_hold, n_repeats=5, random_state=42, n_jobs=-1)
importance = pd.Series(perm.importances_mean, index=feat_cols).sort_values(ascending=False)

importance

HIST_BLK           0.246777
TOWN               0.098344
FLOOR_AREA_SQM     0.082078
FLAT_TYPE          0.065648
FLAT_MODEL         0.011400
LEASE_AGE_YR       0.007575
DIST_MRT_KM        0.006360
DIST_MALL_KM       0.001348
HIST_BLK_N         0.001025
COE_PREMIUM_AVG    0.000320
MTH_OF_YR          0.000148
STOCK_CLOSE_AVG    0.000097
DIST_SCH_KM        0.000097
MTH_SINCE_START    0.000000
dtype: float64

In [35]:
#REFIT ON FULL LABELED DATA + PREDICT TEST + SAVE SUBMISSION
#WHY: STD KAGGLE PRACTICE - ONCE HYPERPARAMS CHOSEN, REFIT ON 100% OF LABELED ROWS BEFORE PREDICTING REAL TEST SET, NO ROWS WASTED ON FINAL MODEL
#HISTORY SOURCE IS NOW ALL OF df_train - SAME FROZEN-CUTOFF RULE THE HOLDOUT USED, JUST WITH MORE ROWS BEHIND IT
X_train_full, y_train_full = build_X(df_train, df_train, False), df_train['MONTHLY_RENT']
X_test = build_X(df_test, df_train, True)

final_model = HistGradientBoostingRegressor(categorical_features='from_dtype', random_state=42, **search.best_params_)
final_model.fit(X_train_full, y_train_full)
test_pred = final_model.predict(X_test)

submission = pd.DataFrame({'Id': range(len(test_pred)), 'Predicted': test_pred})
submission.to_csv('submissions/histgb_blockhist.csv', index=False)
submission.head()

,Id,Predicted
0,0,2881.175895
1,1,3743.393854
2,2,3331.976100
3,3,3243.986842
4,4,3485.971538
